In [0]:
import requests, pandas as pd
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.solar_pump")

talukas = [("Junnar",19.21,73.88),("Khed",18.86,73.89),
           ("Baramati",18.15,74.58),("Indapur",18.11,75.02)]

def fetch(name, lat, lon):
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point",
        params={"parameters":"ALLSKY_SFC_SW_DWN","community":"AG","latitude":lat,
                "longitude":lon,"format":"JSON","start":"20200101","end":"20241231"},
        timeout=120)
    d = r.json()["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
    df = pd.DataFrame({"date":list(d.keys()),"ghi":list(d.values())})
    df["taluka"] = name
    return df

raw = pd.concat([fetch(*t) for t in talukas], ignore_index=True)
spark.createDataFrame(raw).write.mode("overwrite").saveAsTable("workspace.solar_pump.bronze_solar")
print(raw.shape)

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.solar_pump.silver_solar")

spark.sql("""
CREATE TABLE workspace.solar_pump.silver_solar AS
SELECT taluka,
       to_date(date, 'yyyyMMdd') AS date,
       ghi / 3.6 AS ghi
FROM workspace.solar_pump.bronze_solar
WHERE ghi > -900
""")

display(spark.sql("""
SELECT s.taluka,
       count(*) AS days,
       round(avg(s.ghi), 2) AS avg_ghi_kwh,
       round(avg(s.ghi) * 3.6, 2) AS avg_ghi_mj_raw
FROM workspace.solar_pump.silver_solar s
GROUP BY s.taluka
"""))

In [0]:
A = dict(array_kwp=5.0, performance_ratio=0.75, pump_kw=3.7, hours_per_day=6,
         irrigation_days=120, diesel_l_per_hour=1.2, diesel_price=90,
         solar_capex=300000, subsidy_pct=0.60, om_per_year=3000)

spark.sql("""
CREATE OR REPLACE TABLE workspace.solar_pump.gold_payback AS
SELECT taluka,
       round(avg(ghi),2) AS avg_ghi,
       round(avg(ghi) * {kwp} * {pr}, 1) AS kwh_per_day
FROM workspace.solar_pump.silver_solar GROUP BY taluka
""".format(kwp=A["array_kwp"], pr=A["performance_ratio"]))

g = spark.table("workspace.solar_pump.gold_payback").toPandas()
need = A["pump_kw"] * A["hours_per_day"]
g["solar_coverage_pct"] = (g.kwh_per_day / need * 100).clip(upper=100).round(0)
diesel_cost = A["diesel_l_per_hour"]*A["hours_per_day"]*A["irrigation_days"]*A["diesel_price"]
g["annual_diesel_cost"] = round(diesel_cost)
g["annual_saving"] = round(diesel_cost * g.solar_coverage_pct/100 - A["om_per_year"])
g["farmer_capex"] = A["solar_capex"] * (1 - A["subsidy_pct"])
g["payback_years"] = (g.farmer_capex / g.annual_saving).round(1)
g["payback_no_subsidy"] = (A["solar_capex"] / g.annual_saving).round(1)

spark.createDataFrame(g).write.mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("workspace.solar_pump.gold_payback")
display(g)